# data load

In [41]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.combine import SMOTETomek
from imblearn.ensemble import EasyEnsembleClassifier



In [17]:
# 현재 노트북 위치
BASE_DIR = Path().resolve()

# 프로젝트 루트 (notebooks의 상위 폴더)
PROJECT_ROOT = BASE_DIR.parent

# data 폴더
DATA_DIR = PROJECT_ROOT / "data"

# data 파일
path1 = DATA_DIR / "all_data_v1_update.csv"
trade_area_path = DATA_DIR / "서울시 상권분석서비스(길단위인구-상권).csv"

# data load
df = pd.read_csv(path1, encoding="utf-8")
trade_area_info = pd.read_csv(trade_area_path, encoding="cp949")

# 파생변수 생성

In [18]:
# -----------------------
# 1) 성별 컬럼 정의
# -----------------------
female_cols = [c for c in df.columns if c.startswith("여성")]
male_cols   = [c for c in df.columns if c.startswith("남성")]

# -----------------------
# 2) 행 단위 성별 합계
# -----------------------
df["여성_합계"] = df[female_cols].sum(axis=1)
df["남성_합계"] = df[male_cols].sum(axis=1)


In [19]:
# -----------------------
# 1) 연령대 컬럼 정의
# -----------------------
age_cols = {
    "20대": [c for c in df.columns if "20대" in c],
    "30대": [c for c in df.columns if "30대" in c],
    "40대": [c for c in df.columns if "40대" in c],
    "50대": [c for c in df.columns if "50대" in c],
    "60대": [c for c in df.columns if "60대" in c],

}

# -----------------------
# 2) 행(row) 단위 연령대 합계
# -----------------------
for age, cols in age_cols.items():
    df[f"{age}_합계"] = df[cols].sum(axis=1)

In [20]:
df["배달여부"] = (
    df.groupby("가맹점구분번호")["배달매출금액 비율"]
      .transform(lambda x: (x != 0).any())
      .astype(int)
)

# 분기별 요약변수 생성

In [21]:
# 분기 컬럼 생성
df["기준년월"] = pd.to_datetime(df["기준년월"])
df["연도"] = df["기준년월"].dt.year
df["분기"] = df["기준년월"].dt.quarter

In [22]:
target_cols = ['가맹점 운영개월수 구간','배달여부', '배달매출금액 비율','객단가 구간', '동일 업종 내 해지 가맹점 비중',
               '여성_합계', '남성_합계',
               '20대_합계', '30대_합계', '40대_합계', '50대_합계','60대_합계',]


In [23]:
# 상점×분기 압축 (평균/표준편차)
summary_df = (
    df.groupby(["가맹점구분번호", "연도", "분기"], dropna=False)[target_cols]
      .agg(["mean", "std"])
)
# 컬럼명 깔끔하게: (컬럼, mean/std) → "컬럼_mean" 형태
summary_df.columns = [f"{col}_{stat}" for col, stat in summary_df.columns]
summary_df = summary_df.reset_index()

# 선택: 표준편차가 NaN(분기 데이터 1개뿐이면 std가 NaN)인 경우 0으로
std_cols = [c for c in summary_df.columns if c.endswith("_std")]
summary_df[std_cols] = summary_df[std_cols].fillna(0)


In [24]:
summary_df.head()

,가맹점구분번호,연도,분기,가맹점 운영개월수 구간_mean,가맹점 운영개월수 구간_std,배달여부_mean,배달여부_std,배달매출금액 비율_mean,배달매출금액 비율_std,객단가 구간_mean,...,20대_합계_mean,20대_합계_std,30대_합계_mean,30대_합계_std,40대_합계_mean,40대_합계_std,50대_합계_mean,50대_합계_std,60대_합계_mean,60대_합계_std
0,000F03E44A,2023,1,5.000000,0.00000,1.0,0.0,69.283333,0.0,5.333333,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,0.000000,0.00000
1,000F03E44A,2023,2,5.000000,0.00000,1.0,0.0,69.283333,0.0,5.000000,...,0.000000,0.000000,63.900000,12.733028,0.000000,0.000000,36.100000,12.733028,0.000000,0.00000
2,000F03E44A,2023,3,5.000000,0.00000,1.0,0.0,69.283333,0.0,5.666667,...,0.000000,0.000000,70.000000,8.660254,0.000000,0.000000,30.000000,8.660254,0.000000,0.00000
3,000F03E44A,2023,4,4.333333,0.57735,1.0,0.0,69.283333,0.0,6.000000,...,11.133333,9.641749,53.333333,5.773503,0.000000,0.000000,35.533333,3.868247,0.000000,0.00000
4,000F03E44A,2024,1,4.000000,0.00000,1.0,0.0,69.283333,0.0,5.333333,...,23.833333,8.247626,45.766667,16.755994,3.033333,5.253887,15.266667,5.589574,12.133333,21.01555


In [25]:
## p-value < 0.05인 변수만 선택
selected_cols = [
    '가맹점구분번호', '연도', '분기',
    '남성_합계_mean',
    '여성_합계_mean',
    '가맹점 운영개월수 구간_mean',
    '배달매출금액 비율_std',
    '60대_합계_mean',
    '20대_합계_mean',
    '배달매출금액 비율_mean',
    '배달여부_mean',
    '30대_합계_mean',
    '50대_합계_mean',
    '객단가 구간_mean',
    '60대_합계_std',
    '동일 업종 내 해지 가맹점 비중_std',
    '객단가 구간_std'
]
summary_df = summary_df[selected_cols]


In [26]:
df_first = (
    df[["가맹점구분번호", "폐업여부",'상권']]
    .drop_duplicates(subset="가맹점구분번호", keep="first")
)

final_df = summary_df.merge(
    df_first,
    on="가맹점구분번호",
    how="left"
)


# 파생변수 - 상권 활성화 지수

## 상권_유동인구_수 

In [27]:
# 0) 기준 분기 & 성동구 상권 목록
target_quarters = [20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244]
seongdong_area = df["상권"].unique()

# 1) 외부 유동인구 데이터 전처리 (필터링 + 연도/분기 분리 + 컬럼 정리)
trade_area_population = (
    trade_area_info.loc[
        trade_area_info["기준_년분기_코드"].isin(target_quarters)
        & trade_area_info["상권_코드_명"].isin(seongdong_area),
        ["상권_코드_명", "기준_년분기_코드", "총_유동인구_수"]
    ]
    .assign(
        연도=lambda x: x["기준_년분기_코드"].astype(str).str[:4].astype(int),
        분기=lambda x: x["기준_년분기_코드"].astype(str).str[4:].astype(int),
        상권=lambda x: x["상권_코드_명"]
    )
    .drop(columns=["상권_코드_명", "기준_년분기_코드"])
    .rename(columns={"총_유동인구_수": "상권_유동인구_수"})
)

## 상권_소비력

In [28]:
# 1) 상권별 평균값 집계
area_agg = (
    df.groupby("상권", as_index=False)
        .agg({
            "매출금액 구간": "mean",
            "유니크 고객 수 구간": "mean",
        })
)

# # 2) 순위/구간 점수화
#    - 매출금액, 고객 수 구간: 1~6 → 0~100 선형 변환
area_agg["매출_점수"] = (6 - area_agg["매출금액 구간"]) / 5 * 100
area_agg["고객수_점수"] = (6 - area_agg["유니크 고객 수 구간"]) / 5 * 100


# 3) 상권 소비력 계산
#    - 매출, 고객 수, 업종 내 순위 점수 평균

sel_col = ["매출_점수", "고객수_점수"] 
area_agg["상권_소비력"] = area_agg[sel_col].mean(axis=1)
area_active= trade_area_population.merge(area_agg[["상권","상권_소비력"]])

## 상권_활성화지수

In [29]:
# 1) 표준화
scaler = StandardScaler()
scaled = scaler.fit_transform(area_active[["상권_유동인구_수", "상권_소비력"]])

# 2) 음수값 처리 및 기하평균 계산
scaled_shifted = scaled - scaled.min(axis=0) + 1e-9
area_active["상권_활성화지수"] = np.sqrt(scaled_shifted[:, 0] * scaled_shifted[:, 1])
area_active = area_active.drop(columns=["상권_유동인구_수", "상권_소비력"])

In [32]:
# 2) 병합 (상권/연도/분기 기준)
merged_df = final_df.merge(
    area_active,
    on=["상권", "연도", "분기"],
    how="left"
)
# 결측치 처리
merged_df["상권_활성화지수"] = merged_df["상권_활성화지수"].fillna(
    merged_df["상권_활성화지수"].mean()
)
merged_df = merged_df.drop(columns=["상권"])

# 
final_df = merged_df.copy()


# ex) 2024년 1분기 폐업 예측

In [33]:
df_2023_q1 = final_df[(final_df["연도"] == 2024) & (final_df["분기"] == 1)]

X = df_2023_q1.drop(columns=['가맹점구분번호','연도','분기','폐업여부'])
y = df_2023_q1["폐업여부"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

#  표준화
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# # SMOTE 적용
# smote = SMOTETomek(random_state=42)
# X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

In [34]:
# 모델 정의
models = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    'SVM': SVC(random_state=42, class_weight="balanced"),
    'EasyEnsembleClassifier': EasyEnsembleClassifier(random_state=47, n_jobs=-1)
}


# 결과 저장
results = {'Model': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []}

# 모델별 학습 및 평가
for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results['Model'].append(model_name)
    results['Accuracy'].append(accuracy_score(y_test, y_pred))
    results['Precision'].append(precision_score(y_test, y_pred))
    results['Recall'].append(recall_score(y_test, y_pred))
    results['F1'].append(f1_score(y_test, y_pred))

# 결과를 DataFrame으로 변환
results_df = pd.DataFrame(results)

In [35]:
results_df

,Model,Accuracy,Precision,Recall,F1
0,LogisticRegression,0.623644,0.031519,0.55,0.059621
1,SVM,0.777657,0.010582,0.10,0.019139
2,EasyEnsembleClassifier,0.542299,0.028169,0.60,0.053812


# Baseline_LogisticRegression

In [36]:
# 2) 앙상블 평가용 공통 test set 1번만 분리
X_all = final_df.drop(columns=['가맹점구분번호','연도','분기','폐업여부'])
y_all = final_df["폐업여부"].astype(int)

X_train_all, X_test, y_train_all, y_test = train_test_split(
    X_all, y_all, stratify=y_all, random_state=42, test_size=0.2
)

test_idx = X_test.index  # 누수 방지용(분기 train에서 test행 제외)

# 3) 8개(연도×분기) 모델 학습
models = []  # (scaler, model) 저장

for year in [2023, 2024]:
    for q in [1, 2, 3, 4]:
        df_q = final_df[(final_df["연도"] == year) & (final_df["분기"] == q)].copy()

        # 분기 데이터에서 test에 들어간 행 제거(누수 방지)
        df_q = df_q.drop(index=test_idx, errors="ignore")

        # X,y 만들기
        X_q = df_q.drop(columns=['가맹점구분번호','연도','분기','폐업여부'])
        y_q = df_q["폐업여부"].astype(int)

        # 학습 불가(클래스 한쪽뿐/데이터 너무 적음)면 스킵
        if y_q.nunique() < 2 or len(y_q) < 30:
            print(f"[SKIP] {year} Q{q} (데이터/클래스 부족)")
            continue

        # 스케일링(분기 train 기준)
        scaler = StandardScaler()
        X_q_scaled = scaler.fit_transform(X_q)

        # SMOTETomek(분기 train에만)
        smt = SMOTETomek(random_state=42)
        X_q_bal, y_q_bal = smt.fit_resample(X_q_scaled, y_q)

        # 모델 학습
        model = LogisticRegression(random_state=42, max_iter=1000)
        model.fit(X_q_bal, y_q_bal)

        models.append((scaler, model))
        print(f"[OK] {year} Q{q} model trained")

print("학습된 분기 모델 수:", len(models))

# 4) 앙상블(soft voting): 분기 모델들의 예측확률 평균
probas = []
for scaler, model in models:
    X_test_scaled = scaler.transform(X_test)
    probas.append(model.predict_proba(X_test_scaled)[:, 1])

avg_proba = np.mean(np.vstack(probas), axis=0)
y_pred = (avg_proba >= 0.5).astype(int)

# 5) 평가
results_df = pd.DataFrame([{
    "Model": f"Quarter Ensemble ({len(models)} models)",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0)
}])

results_df


[OK] 2023 Q1 model trained
[OK] 2023 Q2 model trained
[OK] 2023 Q3 model trained
[OK] 2023 Q4 model trained
[OK] 2024 Q1 model trained
[OK] 2024 Q2 model trained
[OK] 2024 Q3 model trained
[OK] 2024 Q4 model trained
학습된 분기 모델 수: 8


,Model,Accuracy,Precision,Recall,F1
0,Quarter Ensemble (8 models),0.581395,0.027414,0.558333,0.052262


# 여러 모델 비교 

In [42]:
# -------------------------------------------------
# 0) 공통 테스트셋 1번만 분리
# -------------------------------------------------
df = final_df.copy()

X_all = df.drop(columns=['가맹점구분번호','연도','분기','폐업여부'])
y_all = df["폐업여부"].astype(int)

X_train_all, X_test, y_train_all, y_test = train_test_split(
    X_all, y_all, stratify=y_all, random_state=42, test_size=0.2
)
test_idx = X_test.index   # 누수 방지: 분기 train에서 test 행 제거


# -------------------------------------------------
# 1) 비교할 모델 후보(이름, 모델객체, SMOTE쓸지)
# -------------------------------------------------
candidates = [
    ("LogReg + SMOTETomek",
     LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
     True),

    ("SVM + SMOTETomek",
     SVC(random_state=42, class_weight="balanced", probability=True),
     True),

    ("BalancedRandomForest (no SMOTE)",
     BalancedRandomForestClassifier(random_state=42, n_estimators=300, n_jobs=-1),
     False),

    ("EasyEnsemble (no SMOTE)",
     EasyEnsembleClassifier(random_state=47, n_jobs=-1),
     False),

    ("RandomForest (class_weight, no SMOTE)",
     RandomForestClassifier(random_state=42, n_estimators=300, class_weight="balanced", n_jobs=-1),
     False),
]


# -------------------------------------------------
# 2) 후보 모델 1개에 대해 "8개 분기 모델 앙상블" 평가하는 함수
# -------------------------------------------------
def eval_quarter_ensemble(base_model, use_smote, min_rows=30):
    trained = []  # (scaler, model)

    for year in [2023, 2024]:
        for q in [1, 2, 3, 4]:
            df_q = df[(df["연도"] == year) & (df["분기"] == q)].copy()
            df_q = df_q.drop(index=test_idx, errors="ignore")  # 누수 방지

            X_q = df_q.drop(columns=['가맹점구분번호','연도','분기','폐업여부'])
            y_q = df_q["폐업여부"].astype(int)

            # 학습 불가 방지(클래스 1개면 SMOTE도 불가)
            if y_q.nunique() < 2 or len(y_q) < min_rows:
                continue

            scaler = StandardScaler()
            X_q_scaled = scaler.fit_transform(X_q)

            if use_smote:
                smt = SMOTETomek(random_state=42)
                X_q_scaled, y_q = smt.fit_resample(X_q_scaled, y_q)

            # 모델은 분기마다 새로 생성해야 함(객체 재사용하면 덮어씀)
            model = type(base_model)(**base_model.get_params())
            model.fit(X_q_scaled, y_q)

            trained.append((scaler, model))

    if len(trained) == 0:
        return None

    # soft voting: 예측확률 평균
    probas = []
    for scaler, model in trained:
        X_test_scaled = scaler.transform(X_test)
        probas.append(model.predict_proba(X_test_scaled)[:, 1])

    avg_proba = np.mean(np.vstack(probas), axis=0)
    y_pred = (avg_proba >= 0.5).astype(int)

    return {
        "n_models_used": len(trained),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    }


# -------------------------------------------------
# 3) 후보별로 돌려서 결과 비교
# -------------------------------------------------
rows = []
for name, model, use_smote in candidates:
    out = eval_quarter_ensemble(model, use_smote=use_smote, min_rows=30)
    if out is None:
        rows.append({"Model": name, "n_models_used": 0, "Accuracy": np.nan, "Precision": np.nan, "Recall": np.nan, "F1": np.nan})
    else:
        rows.append({"Model": name, **out})

results_df = pd.DataFrame(rows).sort_values("F1", ascending=False)
results_df


,Model,n_models_used,Accuracy,Precision,Recall,F1
2,BalancedRandomForest (no SMOTE),8,0.844961,0.060811,0.450000,0.107143
1,SVM + SMOTETomek,8,0.958312,0.076389,0.091667,0.083333
3,EasyEnsemble (no SMOTE),8,0.499397,0.028755,0.708333,0.055267
0,LogReg + SMOTETomek,8,0.581395,0.027414,0.558333,0.052262
4,"RandomForest (class_weight, no SMOTE)",8,0.979328,0.000000,0.000000,0.000000
